In [ ]:
# Load packages
from pathlib import Path
import os
import re
import sys
import importlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from skyline_qc import *

In [ ]:

# Skyline data path
skyline_path = "data/MARTHA/DE17501_Martha_results_subset_07042026.csv"

# Import skyline data (includes column "Isotope Label Type" from Precursor)
skyline_importer = ImportFile(skyline_path)
skyline_data = skyline_importer.import_skyline_file()

In [ ]:
# Isotope Label Type is added in ImportFile.import_skyline_file() (from Precursor).

# qREPs spike levels
qREPs_spike_levels = pd.read_csv('ratio/DE17501_ratio.csv')

# SDRF
sdrf_path = 'sdrf/sdrf_MARTHA_pool.sdrf.tsv'
sdrf_data = pd.read_csv(sdrf_path, sep='\t')



In [ ]:
# Check file sets match

cross_check_skyline_sdrf(skyline_df = skyline_data, sdrf_df = sdrf_data)

In [ ]:
# Import and use function from output_test.py to get iRT peptides
iRT_peptides = get_irt_peptides(skyline_data)

print('This is the iRT peptides:')
print(iRT_peptides)

In [ ]:
skyline_data.head()

In [ ]:
plot_library_dot_product_distribution(skyline_data)

In [ ]:
skyline_data.head()

#Group by Peptide, summarise average of Library Dot Product and sort from highest to lowest
skyline_data_peptide_avg = skyline_data.groupby(['Peptide'])['Library Dot Product'].mean().sort_values(ascending=False)
skyline_data_peptide_avg.head(20)

# #Filter to keep only peptides with average Library Dot Product greater than 0.8
# skyline_clean = skyline_data[skyline_data['Library Dot Product'] > 0.8]


In [ ]:
skyline_data.head# Filter to peptide ISASAEELR from skyline_data
skyline_data_isasaee = skyline_data[skyline_data['Peptide'] == 'ISASAEELR']
skyline_data_isasaee.head()


In [ ]:
skyline_clean = filter_library_dot_product(skyline_data, threshold=0.8)


In [ ]:
# Summarise each peptide to count how many heavy or light signals are present in skyline_pivot
peptide_counts = summarise_peptide_counts(skyline_clean)


In [ ]:
report_summary, peptide_list = report_peptide_protein_summary(peptide_counts)

In [ ]:
plot_heavy_light_scatter(peptide_counts)

In [ ]:
filtered_peptide_counts = filter_peptide_counts(peptide_counts, light_cutoff=10, heavy_cutoff=10)

filtered_peptide_counts.head()


In [ ]:
selected_peptides_report,selected_peptides = report_peptide_protein_summary(filtered_peptide_counts)

In [ ]:
from skyline_qc.importer import MergeFiles

skyline_merge = MergeFiles(skyline_data, sdrf_data, selected_peptides).merge_files()


In [ ]:
skyline_merge.head()

In [ ]:
# Batch/Plate adjustment, correcting for each peptide within each plate (batch) 

pool_data = skyline_merge[skyline_merge['characteristics[Sample]'] == 'Pool']
# First, sort dataframe by Plate
pool_data = pool_data.sort_values('characteristics[Plate]')
# Reset index
pool_data = pool_data.reset_index(drop=True)
pool_data.head()

In [ ]:
# Plot log_ratio of each sample in boxplot, colored by plate, but x-axis is Replicate, sorted by plate (though x labels are hidden).
plot_pool_boxplot(pool_data)


In [ ]:
# Calculate intra-plate CV

peptide_plate_stats = calculate_intra_plate_cv(pool_data, col_name='characteristics[Plate]')
plot_intra_plate_cv_stats(peptide_plate_stats, col_name='characteristics[Plate]')

In [ ]:
# Example usage:
interplate_cv = calculate_inter_plate_cv(peptide_plate_stats)
interplate_cv.head()

In [ ]:
plot_cumulative_peptide_count_by_cv(interplate_cv)

# OpenSWATH



Can you filtering the results by different scores (eg the `VAR_LIBRARY_DOTPROD`, this should be similar / the same as Skylines dotp), and compare the identified RT apex and Intensity with Skylines. You can probably remove the decoys for this. So do

1. Remove all decoys
2. filter each runs precursor by the highest VAR_LIBRARY_DOTPROD
3. filter the remaining results for VAR_LIBRARY_DOTPROD > some threshold (i.e. 0.90 or an equivalent threshold you usually use in SKyline)


In [ ]:
# Import OpenSWATH results

from openms_qc import *

In [ ]:
# OpenMS results from Justin
openms_path = 'openms/openswath_results_export.tsv'

openswath_df = import_openswath_file(openms_path, remove_file_path=True)


In [ ]:
openswath_df.head()

In [ ]:
openswath_filtered = filter_best_peak_group(openswath_df, threshold=0.0)
# Fix the thresdhol later

In [ ]:
# plot dotprod kde
plot_dotprod_kde(openswath_filtered)

In [ ]:
from openms_qc.openswath import _summarise_ions_channel_count, _summarise_ions_channel, _select_ions_channel

In [ ]:
count_ions_channel(openswath_filtered)

In [ ]:
from openms_qc.openswath import _summarise_ions_channel_count, _summarise_ions_channel
_summarise_ions_channel_count(openswath_filtered)

In [ ]:

plot_ions_channel(openswath_filtered)

In [ ]:
count_ions_channel(openswath_filtered)

In [ ]:

selected_peptides_df = filter_ions_channel(openswath_filtered, light_cutoff=10, heavy_cutoff=10)
selected_peptides_df

# Match Skyline and OpenSWATH

In [ ]:
openswath_ratio = get_ratio(selected_peptides_df, level='peptide')


In [ ]:
openswath_ratio_simple = openswath_ratio[['filename', 'Sequence', 'ratio_light_to_heavy']]
# Rename to skyline_ratio
openswath_ratio_simple.rename(columns={'filename': 'File Name', 'Sequence': 'Peptide', 'ratio_light_to_heavy': 'openswath_ratio'}, inplace=True)
# Sort by File Name and Peptide
openswath_ratio_simple = openswath_ratio_simple.sort_values(['File Name', 'Peptide'])
# filter to overlap peptidesio_simple.sort_values(['File Name', 'Peptide'])


skyline_merge_simple = skyline_merge[['File Name', 'Peptide', 'RatioLightToHeavy']]
# Rename to skyline_ratio
skyline_merge_simple.rename(columns={'RatioLightToHeavy': 'skyline_ratio'}, inplace=True)
# Sort by File Name and Peptide
skyline_merge_simple = skyline_merge_simple.sort_values(['File Name', 'Peptide'])
# In column 'File Name', replae ending .raw with .mzML
skyline_merge_simple['File Name'] = skyline_merge_simple['File Name'].str.replace('.raw', '.mzML')
# Drop duplicates
skyline_merge_simple = skyline_merge_simple.drop_duplicates(subset=['File Name', 'Peptide'])
# Reset index
skyline_merge_simple = skyline_merge_simple.reset_index(drop=True)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

# Get unique peptide sets
openswath_peptides = set(openswath_ratio_simple['Peptide'].dropna().unique())
skyline_peptides = set(skyline_merge_simple['Peptide'].dropna().unique())

# Calculate overlap and unique peptides
overlap_peptide = openswath_peptides & skyline_peptides
openswath_not_in_skyline = openswath_peptides - skyline_peptides
skyline_not_in_openswath = skyline_peptides - openswath_peptides

# Plot Venn diagram
plt.figure(figsize=(6,4))
venn2([openswath_peptides, skyline_peptides], set_labels=('OpenSWATH', 'Skyline'))
plt.title('Venn Diagram of Peptide Overlap')
plt.show()


In [ ]:
# Left join openswath_ratio_simple and skyline_merge_simple by File Name and Peptide
# Left join on both 'File Name' and 'Peptide'
merge_df = pd.merge(
    # Filter to overlap peptides for both openswath_ratio_simple and skyline_merge_simple
    openswath_ratio_simple[openswath_ratio_simple['Peptide'].isin(overlap_peptide)],
    skyline_merge_simple[skyline_merge_simple['Peptide'].isin(overlap_peptide)],
    on=['File Name', 'Peptide'],
    how='left'
)

# Filter Ratio to be within 10^-3 and 10^3
merge_df = merge_df[(merge_df['openswath_ratio'] >= 10**-3) & (merge_df['openswath_ratio'] <= 10**3)]
merge_df = merge_df[(merge_df['skyline_ratio'] >= 10**-3) & (merge_df['skyline_ratio'] <= 10**3)]

# Extract Plate from File Name wiht a number after Plate_
merge_df['Plate'] = merge_df['File Name'].str.extract(r'Plate_(\d+)')

In [ ]:
# Write a function where it plots the dot plot of openswath_ratio vs skyline_ratio, colored by Plate
def plot_openswath_skyline_ratio(merge_df, peptide):
    # Filter merge_df for the given peptide
    peptide_df = merge_df[merge_df['Peptide'] == peptide]
    
    # Choose a color map for plate coloring
    import matplotlib.cm as cm
    import matplotlib.colors as mcolors

    # Handle Plate as categorical
    plates = peptide_df['Plate'].astype(str)
    unique_plates = sorted(plates.unique())
    plate_to_num = {plate: idx for idx, plate in enumerate(unique_plates)}
    colors = [cm.tab10(plate_to_num[plate] % 10) for plate in plates]

    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(
        peptide_df['openswath_ratio'],
        peptide_df['skyline_ratio'],
        alpha=0.7,
        c=colors,
        label=None
    )
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel('OpenSWATH Ratio')
    plt.ylabel('Skyline Ratio')
    plt.plot([0, 10], [0, 10], color='black', linestyle='--')

    # Create a legend with plate numbers and their color
    import matplotlib.patches as mpatches
    handles = [
        mpatches.Patch(color=cm.tab10(plate_to_num[plate] % 10), label=f"Plate {plate}")
        for plate in unique_plates
    ]
    plt.legend(handles=handles, title='Plate')

    plt.title(f'OpenSWATH vs Skyline Ratio for {peptide}')
    # plt.show()

# Example usage
plot_openswath_skyline_ratio(merge_df, 'ACIPTGPYPCGK')


In [ ]:
# Plot every peptide and save all plots to a single PDF file using plot_openswath_skyline_ratio
from matplotlib.backends.backend_pdf import PdfPages
import os

output_dir = "plot"
output_path = os.path.join(output_dir, "openswath_skyline_ratio.pdf")
os.makedirs(output_dir, exist_ok=True)

with PdfPages(output_path) as pdf:
    for peptide in merge_df['Peptide'].unique():
        plot_openswath_skyline_ratio(merge_df, peptide)
        pdf.savefig(plt.gcf())
        plt.close()

# Check RT

In [ ]:
skyline_rt = skyline_data[['File Name', 'Peptide', 'Peptide Retention Time']]

skyline_rt.head()

In [ ]:
skyline_rt.head()

In [ ]:
openswath_rt = openswath_df.copy()
# openswath_rt = openswath_df[['filename', 'Sequence', 'RT']]

# Filter prak_group_rank ==1
openswath_rt = openswath_rt[openswath_rt['peak_group_rank'] == 1]
# Rename filename to File Name and Sequence to Peptide
openswath_rt.rename(columns={'filename': 'File Name', 'Sequence': 'Peptide'}, inplace=True)
# Replace .mzML with .raw in File Name
openswath_rt['File Name'] = openswath_rt['File Name'].str.replace('.mzML', '.raw')

# Remove modification from FullPeptideName
mod_list = ['UniMod:1', 'UniMod:35', 'UniMod:4']
# Use regex to Filter out mod_list from FullPeptideName, I meant remove the whole row if it contains any of the mod_list
openswath_rt = openswath_rt[~openswath_rt['FullPeptideName'].str.contains('|'.join(mod_list))]

# Remove modification from FullPeptideName
mod_list = ['UniMod:1', 'UniMod:35', 'UniMod:4']
# Use regex to Filter out mod_list from FullPeptideName, I meant remove the whole row if it contains any of the mod_list
openswath_rt['Peptide'] = openswath_rt['FullPeptideName'].str.replace(r'\(UniMod:1\)$', '', regex=True)
openswath_rt['Peptide'] = openswath_rt['FullPeptideName'].str.replace(r'\(UniMod:35\)$', '', regex=True)
openswath_rt['Peptide'] = openswath_rt['FullPeptideName'].str.replace(r'\(UniMod:4\)$', '', regex=True)

# Unique peptide from merge_df
unique_peptides = merge_df['Peptide'].unique()

# Filter openswath_rt for unique peptides
openswath_rt = openswath_rt[openswath_rt['Peptide'].isin(unique_peptides)]



openswath_rt.head()



In [ ]:
# # Left join skyline_rt and openswath_rt by File Name and Peptide
merge_rt = pd.merge(skyline_rt, openswath_rt, on=['File Name', 'Peptide'], how='inner')


merge_rt.head()


In [ ]:
import matplotlib.pyplot as plt

# Get all unique File Names
file_names = merge_rt['File Name'].unique()
file_names = sorted(file_names)  # ensure consistent order

# Set up 5x5 grid for up to 25 unique files
n_rows, n_cols = 5, 5
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 20), sharex=True, sharey=True)
axes = axes.flatten()

for idx, file_name in enumerate(file_names[:n_rows * n_cols]):
    ax = axes[idx]
    subset = merge_rt[merge_rt['File Name'] == file_name]
    ax.scatter(subset['RT'], subset['Peptide Retention Time'], alpha=0.7)
    ax.set_title(file_name, fontsize=8)
    if idx % n_cols == 0:
        ax.set_ylabel('Skyline RT')
    if idx // n_cols == n_rows - 1:
        ax.set_xlabel('OpenSWATH RT')

# Hide extra subplots if less than 25
for idx in range(len(file_names), n_rows * n_cols):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.show()


In [ ]:
openswath_df['FullPeptideName'].head()